# step04_suna_time_lag - SUNA / CTD time alignment diagnostic

**Question (Pierre Marrec, 11 Nov 2026):** is the SUNA nitrate shifted in time relative to the CTD, and is that
a clock-synchronisation problem? He asked for (1) the clock situation, (2) nitrate and depth plotted against time
with the lag between them computed, and (3) a photo of how the SUNA is mounted on the rosette.

**What this notebook does, per cast and for the cruise:**

1. Estimates the SUNA-to-CTD lag with three methods, each tested first on a synthetic cast whose lag is known.
2. Gives each cast's lag a 95% confidence interval, and the cruise a median with its own interval.
3. Tests for clock drift through the cruise (a drifting clock gives a lag that changes steadily with time).
4. Runs an independent clock check from the moment the rosette enters and leaves the water, which does not
   depend on the nitrate profile at all.
5. Shows the offset: static figures per cast, a cruise summary figure, and an interactive window with a
   slider that shifts the SUNA time so you can watch the downcast and upcast collapse onto each other.
6. Reports the `SUNA_CLOCK_OFFSET_S` value, with its sign, for step02.

**Sign convention (used everywhere):** a **positive lag means the SUNA reports nitrate later than the CTD
pressure it belongs to** (the SUNA lags). The nitrate stamped at time *t* belongs to the pressure at *t - lag*.
Both step02 notebooks *add* their offset to the SUNA time, so the setting is **minus** the lag.

**Two modes (`SOURCE`), one loop:**

| Step | Where | What |
|---|---|---|
| 1. Measure | this notebook, `SOURCE = "colocated"` | lag on the uncorrected co-located product; writes `<CRUISE>__suna_time_offsets_PROPOSED.csv` |
| 2. Confirm | you | review the proposal, save it as `cruises/<CRUISE>/metadata/suna_time_offsets.csv` |
| 3. Apply | `step02_merge_ctd_suna`, `TIME_OFFSET_MODE = "table"` | shifts SUNA time (and its QC flags) per cast during the merge |
| 4. Verify | this notebook, `SOURCE = "merged"` | lag on step02's corrected output; should be about zero |

**Inputs (read only):** measure mode reads `L2/SUNA/<CRUISE>__suna_L2_colocated.csv` (keep
`step02_suna_colocation` at zero offset so this stays the raw measurement) plus, for the event check, the L1
CTD `.cnv` files and L1 SUNA QC tables (`f_outofwater`); verify mode reads step02's `L2/SUNA/*_SUNA_1s.csv`.
**Outputs:** figures and CSVs under `_audit/suna/time_lag/` (verify mode: `_audit/suna/time_lag/residual/`)
only. This notebook changes no existing product.

## Method and statistics

**Why a time lag shows up as down/up hysteresis.** If the SUNA lags by *L* seconds, each nitrate value is
assigned to the pressure the CTD reached *L* seconds later. On the downcast the instrument is still going
down, so the nitrate is placed too deep, by about *w_down x L*. On the upcast it is placed too shallow, by
*w_up x L*. The two legs therefore separate in depth by about *(w_down + w_up) x L*, and the separation is
largest where nitrate changes fastest with depth: the nitracline. A real ocean profile should be nearly the
same going down and coming up, so the lag shows up as a mismatch between the two legs.

**Method C, down/up collapse (primary).** For each candidate lag from -60 to +60 s in 0.5 s steps, each nitrate
frame is re-assigned to the CTD pressure at *t - lag*. The frames are split into down and up legs at the deepest
point and averaged into 1 dbar bins, and the RMS difference between the two legs is computed over the bins
both legs share. The lag with the smallest RMS is the estimate. This is the principle Garau et al. (2011) use
to correct glider CTD thermal lag: choose the correction that minimises the difference between a downcast and
an upcast profile. We also report how much of the zero-shift mismatch the best shift removes.

**Method B, cross-correlation (secondary).** The lag that maximises the Pearson correlation between nitrate
and lagged pressure, the standard lagged-correlation estimator (Thomson and Emery, 2014). It assumes nitrate
is linear in pressure. It is not: nitrate rises through the nitracline and levels off below it. The validation
below shows this biases the estimator, and the direction of the bias depends on the shape of the profile. We
report it for comparison only.

**Method A, peak to peak (what was asked for).** Time of maximum nitrate minus time of the deepest point.
Easy to explain, but when nitrate plateaus at depth the maximum falls on noise. The validation shows this, so
it is reported with that caveat.

**Uncertainty.** Each cast's collapse lag gets a 95% CI from a moving-block bootstrap over the 1 dbar bins
(Künsch, 1989; Efron and Tibshirani, 1993). Bins are resampled in blocks of 10 because neighbouring bins are
correlated, and resampling single bins would understate the uncertainty. The cruise lag is the median across
casts, with a percentile bootstrap CI over casts. The resolution floor is the SUNA frame interval, about 1 s.

**Clock drift.** A clock that drifts gives a lag that changes steadily through the cruise. We fit lag against
cast time (least squares) and report the slope, its standard error and its p-value. A slope consistent with
zero means one constant offset is enough.

**Independent clock check.** The CTD conductivity jumps from about 0 in air to several S/m when the cell floods,
and the SUNA's `f_outofwater` flag flips when its optical path fills. Comparing the two at entry and at exit
measures the clock offset without using the nitrate profile. If the SUNA is mounted higher than the CTD
sensors, it enters later and leaves earlier, so entry and exit are biased in opposite directions. The mean of
the two cancels the mounting height, and half their difference estimates it. If the SUNA draws water through a
plumbed intake rather than an open optical path, its fill and drain delay adds to both, which is one reason
the mounting photo matters.

**What a time shift can and cannot tell you.** Methods A to C all measure the *total effective delay*: clock
offset plus any sensor response time. A first-order sensor response with time constant *tau* looks like a
delay of about *tau* (Bittig et al., 2014). Two things separate the causes: the event check, which sees the
clock, and the share of hysteresis that a pure shift removes, because a response time also smooths the
profile, which a shift cannot undo. A vertical mounting offset gives a constant *depth* offset on both legs,
not hysteresis. It is invisible to methods A to C and has to be measured from the rosette geometry.

**Assumption to keep in mind.** The collapse method assumes both legs sample the same water (Garau et al.,
2011). A cast lasts roughly 10 to 15 minutes, and ship drift, internal waves or patchiness can make the legs
differ for reasons unrelated to the sensor. That residual sets how much of the hysteresis any shift can remove.

In [ ]:
import sys, re
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import matplotlib
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except Exception:
    def display(o): print(o)

# Locate the repo root so project_paths imports from the notebook folder.
for _cand in (Path.cwd(), *Path.cwd().parents):
    if (_cand / "project_paths.py").exists():
        sys.path.insert(0, str(_cand)); break
import project_paths as pp

# ---- settings -------------------------------------------------------------
CRUISE_ID    = "P45_06"   # the only per-cruise switch
SOURCE       = "colocated"  # "colocated": MEASURE the raw lag and propose offsets
                            # "merged":    VERIFY step02's corrected output (residual should be ~0)
DT_S         = 0.5        # lag grid step (s)
MAX_LAG_S    = 60.0       # search window (s)
BIN_DB       = 1.0        # pressure bin for the down/up comparison (dbar)
BLOCK_BINS   = 10         # bootstrap block length (bins)
N_BOOT       = 1000       # bootstrap resamples per cast
USE_CLEAN    = True       # use only spectrally clean frames (L1_clean)
MIN_FRAMES   = 100        # skip casts with fewer usable frames
EVENT_WINDOW_S = 300.0    # max distance between a CTD and a SUNA water event to pair them (s)

if SOURCE not in ("colocated", "merged"):
    raise ValueError(f"SOURCE must be 'colocated' or 'merged', not {SOURCE!r}")
CRUISE = pp.cruise_dir(CRUISE_ID)
COLOCATED = pp.product_dir(CRUISE_ID, "suna_l2") / f"{CRUISE_ID}__suna_L2_colocated.csv"
MERGED = sorted(pp.product_dir(CRUISE_ID, "suna_l2").glob("*_SUNA_1s.csv"))
OUT = CRUISE / "_audit" / "suna" / "time_lag" / ("" if SOURCE == "colocated" else "residual")
OUT.mkdir(parents=True, exist_ok=True)
TAG = "" if SOURCE == "colocated" else "_residual"
print("mode  :", "MEASURE (raw lag)" if SOURCE == "colocated" else "VERIFY (residual after step02 correction)")
print("input :", COLOCATED if SOURCE == "colocated" else f"{len(MERGED)} files L2/SUNA/*_SUNA_1s.csv")
print("output:", OUT)

In [ ]:
# ---- lag estimators (pure numpy) ----

import numpy as np


def _clean_xy(t, y):
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(t) & np.isfinite(y)
    t, y = t[m], y[m]
    o = np.argsort(t)
    return t[o], y[o]


# ---- Method A: peak to peak (the quantity Pierre asked for) -----------------
def peak_lag(t, no3, pres):
    """Time of maximum nitrate minus time of the deepest point (s)."""
    t = np.asarray(t, dtype=float)
    no3 = np.asarray(no3, dtype=float)
    pres = np.asarray(pres, dtype=float)
    if np.isfinite(no3).sum() < 5 or np.isfinite(pres).sum() < 5:
        return np.nan
    return float(t[np.nanargmax(no3)] - t[np.nanargmax(pres)])


# ---- Method B: time-domain cross-correlation --------------------------------
def xcorr_curve(t, no3, pres, dt=0.5, max_lag_s=60.0):
    """Pearson r between nitrate(t) and pressure(t - lag) for each candidate lag.

    Returns (lags, r). Both series are put on a uniform grid at step dt.
    """
    tn, n = _clean_xy(t, no3)
    tp, p = _clean_xy(t, pres)
    lo, hi = max(tn[0], tp[0]), min(tn[-1], tp[-1])
    g = np.arange(lo, hi, dt)
    n = np.interp(g, tn, n)
    lags = np.arange(-max_lag_s, max_lag_s + dt / 2, dt)
    r = np.full(lags.shape, np.nan)
    for i, L in enumerate(lags):
        ps = np.interp(g - L, tp, p, left=np.nan, right=np.nan)   # pressure at t - lag
        m = np.isfinite(ps)
        if m.sum() > 20:
            r[i] = np.corrcoef(n[m], ps[m])[0, 1]
    return lags, r


def xcorr_lag(t, no3, pres, dt=0.5, max_lag_s=60.0):
    lags, r = xcorr_curve(t, no3, pres, dt, max_lag_s)
    if not np.isfinite(r).any():
        return np.nan, np.nan
    i = int(np.nanargmax(r))
    return float(lags[i]), float(r[i])


# ---- Method C: down/up profile collapse (hysteresis minimisation) -----------
def _profiles_after_shift(t, no3, pres, lag, bin_db=1.0):
    """Re-assign each nitrate frame to the pressure at (t - lag), split into down/up by the
    corrected time relative to the deepest point, and bin both legs onto a common pressure grid.
    Returns (bin_centres, down_mean, up_mean)."""
    tp, p = _clean_xy(t, pres)
    t = np.asarray(t, dtype=float)
    no3 = np.asarray(no3, dtype=float)
    tc = t - lag
    pc = np.interp(tc, tp, p, left=np.nan, right=np.nan)
    t_bottom = tp[np.argmax(p)]
    ok = np.isfinite(pc) & np.isfinite(no3)
    down = ok & (tc <= t_bottom)
    up = ok & (tc > t_bottom)
    if down.sum() < 10 or up.sum() < 10:
        return None, None, None
    edges = np.arange(0.0, np.nanmax(pc[ok]) + bin_db, bin_db)
    centres = edges[:-1] + bin_db / 2

    def binmean(mask):
        idx = np.digitize(pc[mask], edges) - 1
        vals = no3[mask]
        keep = (idx >= 0) & (idx < centres.size)
        sums = np.bincount(idx[keep], weights=vals[keep], minlength=centres.size)
        cnt = np.bincount(idx[keep], minlength=centres.size)
        out = np.full(centres.shape, np.nan)
        nz = cnt > 0
        out[nz] = sums[nz] / cnt[nz]
        return out

    return centres, binmean(down), binmean(up)


def collapse_curve(t, no3, pres, dt=0.5, max_lag_s=60.0, bin_db=1.0, min_overlap=10):
    """RMS difference between the downcast and upcast nitrate profiles for each candidate lag.

    The right lag makes the two legs coincide, because a time lag pushes the downcast profile deeper
    and the upcast profile shallower (or the reverse), by roughly descent_speed * lag each.
    Returns (lags, rms, n_overlap_bins).
    """
    lags = np.arange(-max_lag_s, max_lag_s + dt / 2, dt)
    rms = np.full(lags.shape, np.nan)
    nov = np.zeros(lags.shape, dtype=int)
    for i, L in enumerate(lags):
        c, d, u = _profiles_after_shift(t, no3, pres, L, bin_db)
        if c is None:
            continue
        m = np.isfinite(d) & np.isfinite(u)
        nov[i] = int(m.sum())
        if nov[i] >= min_overlap:
            rms[i] = float(np.sqrt(np.mean((d[m] - u[m]) ** 2)))
    return lags, rms, nov


def collapse_lag(t, no3, pres, dt=0.5, max_lag_s=60.0, bin_db=1.0, tol=0.05):
    """Best collapse lag plus a 'near-optimal' interval: all lags whose RMS is within (1+tol) of the
    minimum. Returns dict with lag, lo, hi, rms_min, rms_zero, pct_reduction."""
    lags, rms, _ = collapse_curve(t, no3, pres, dt, max_lag_s, bin_db)
    if not np.isfinite(rms).any():
        return dict(lag=np.nan, lo=np.nan, hi=np.nan, rms_min=np.nan, rms_zero=np.nan, pct_reduction=np.nan)
    i = int(np.nanargmin(rms))
    rmin = rms[i]
    near = lags[np.isfinite(rms) & (rms <= rmin * (1 + tol))]
    i0 = int(np.argmin(np.abs(lags)))
    r0 = rms[i0]
    red = 100.0 * (1 - rmin / r0) if np.isfinite(r0) and r0 > 0 else np.nan
    return dict(lag=float(lags[i]), lo=float(near.min()), hi=float(near.max()),
                rms_min=float(rmin), rms_zero=float(r0), pct_reduction=float(red))


def descent_speed(t, pres):
    """Median descent rate on the downcast (dbar/s, approx m/s)."""
    tp, p = _clean_xy(t, pres)
    ib = int(np.argmax(p))
    if ib < 5:
        return np.nan
    w = np.diff(p[: ib + 1]) / np.diff(tp[: ib + 1])
    w = w[np.isfinite(w) & (w > 0)]
    return float(np.median(w)) if w.size else np.nan


def bootstrap_median_ci(x, n_boot=5000, alpha=0.05, seed=0):
    """Percentile bootstrap CI for the median across casts."""
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size < 2:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    meds = np.median(rng.choice(x, size=(n_boot, x.size), replace=True), axis=1)
    return (float(np.quantile(meds, alpha / 2)), float(np.quantile(meds, 1 - alpha / 2)))


# ---- uncertainty and event check ----

import numpy as np


def collapse_matrix(t, no3, pres, dt=0.5, max_lag_s=60.0, bin_db=1.0):
    """Per-lag, per-bin squared down/up difference on one fixed pressure grid.

    Returns (lags, centres, D2) where D2[i, k] is (down - up)^2 in bin k at lag i (NaN if either leg
    has no data there)."""
    lags = np.arange(-max_lag_s, max_lag_s + dt / 2, dt)
    pmax = np.nanmax(np.asarray(pres, dtype=float))
    centres = np.arange(0.0, pmax + bin_db, bin_db)[:-1] + bin_db / 2
    D2 = np.full((lags.size, centres.size), np.nan)
    for i, L in enumerate(lags):
        c, d, u = _profiles_after_shift(t, no3, pres, L, bin_db)
        if c is None:
            continue
        k = min(c.size, centres.size)
        D2[i, :k] = (d[:k] - u[:k]) ** 2
    return lags, centres, D2


def collapse_lag_ci(t, no3, pres, dt=0.5, max_lag_s=60.0, bin_db=1.0, block=10,
                    n_boot=1000, min_overlap=10, seed=0):
    """Moving-block bootstrap CI for the collapse lag.

    Adjacent 1 dbar bins are correlated (the profile is smooth), so bins are resampled in contiguous
    blocks of `block` bins rather than one at a time; resampling single bins would understate the
    uncertainty. Returns (lag_hat, ci_lo, ci_hi)."""
    lags, centres, D2 = collapse_matrix(t, no3, pres, dt, max_lag_s, bin_db)
    nb = centres.size

    def best(cols):
        sub = D2[:, cols]
        cnt = np.isfinite(sub).sum(axis=1)
        with np.errstate(invalid="ignore", divide="ignore"):
            rms = np.sqrt(np.nansum(sub, axis=1) / cnt)
        rms[cnt < min_overlap] = np.nan
        return lags[int(np.nanargmin(rms))] if np.isfinite(rms).any() else np.nan

    hat = best(np.arange(nb))
    if nb < block * 2:
        return hat, np.nan, np.nan
    rng = np.random.default_rng(seed)
    starts = np.arange(nb - block + 1)
    nblk = int(np.ceil(nb / block))
    est = []
    for _ in range(n_boot):
        s = rng.choice(starts, size=nblk, replace=True)
        cols = (s[:, None] + np.arange(block)[None, :]).ravel()[:nb]
        v = best(cols)
        if np.isfinite(v):
            est.append(v)
    if len(est) < 50:
        return hat, np.nan, np.nan
    return float(hat), float(np.quantile(est, 0.025)), float(np.quantile(est, 0.975))


def ctd_water_events(time_s, cond, pres=None, cond_in=1.0, min_dry_s=10.0):
    """Immersion and exit times of the CTD from conductivity: in air it reads ~0, in seawater it is
    several S/m (or tens of mS/cm), so a threshold of 1 works for either unit. Returns (t_in, t_out)
    as the first and last wet scan. An event is reported only if the record has at least min_dry_s
    of dry scans before (entry) or after (exit) it; otherwise the logger started or stopped already
    in the water and there is no true event, so NaN is returned."""
    time_s = np.asarray(time_s, dtype=float)
    c = np.asarray(cond, dtype=float)
    wet = np.isfinite(c) & (c > cond_in)
    if not wet.any():
        return np.nan, np.nan
    idx = np.flatnonzero(wet)
    t_in, t_out = float(time_s[idx[0]]), float(time_s[idx[-1]])
    if t_in - time_s[0] < min_dry_s:
        t_in = np.nan
    if time_s[-1] - t_out < min_dry_s:
        t_out = np.nan
    return t_in, t_out


def suna_water_transitions(time_s, out_of_water, min_run=5):
    """Debounced times where the SUNA out-of-water flag changes state. Surface waves can make the flag
    flicker, so only runs lasting min_run frames define the dry and wet states. Returns (entries,
    exits) defined as FIRST CONTACT and LAST CONTACT, to match the CTD definition (first and last wet
    scan): an entry is the first frame after a lasting dry run that is followed by a lasting wet run
    (the start of any flicker zone), and an exit is the first frame of a lasting dry run that follows
    a lasting wet run (the end of any flicker zone)."""
    time_s = np.asarray(time_s, dtype=float)
    f = np.asarray(out_of_water).astype(bool)
    o = np.argsort(time_s)
    time_s, f = time_s[o], f[o]
    if f.size == 0:
        return np.array([]), np.array([])
    # run-length encode, keep only runs of at least min_run frames
    change = np.flatnonzero(f[1:] != f[:-1]) + 1
    starts = np.concatenate([[0], change])
    ends = np.concatenate([change, [f.size]])
    runs = [(s, e, f[s]) for s, e in zip(starts, ends) if e - s >= min_run]
    entries, exits = [], []
    for (s0, e0, v0), (s1, e1, v1) in zip(runs[:-1], runs[1:]):
        if v0 and not v1:            # lasting dry -> lasting wet: first contact
            entries.append(time_s[e0])
        elif (not v0) and v1:        # lasting wet -> lasting dry: last contact
            exits.append(time_s[s1])
    return np.array(entries), np.array(exits)


def event_offsets(ctd_in, ctd_out, suna_entries, suna_exits, window_s=300.0):
    """SUNA minus CTD time at immersion and at exit (s), using the nearest SUNA flip within window_s.

    Positive means the SUNA stamps the event LATER than the CTD (SUNA lags), same convention as the
    collapse lag. If the SUNA sits a height h above the CTD sensors, entry is biased by +h/v and exit
    by -h/v, so the MEAN of the two cancels the mounting height and estimates the clock offset (plus
    any intake fill/drain delay if the SUNA is plumbed rather than open-path)."""
    def nearest(arr, t):
        if not np.isfinite(t) or len(arr) == 0:
            return np.nan
        j = int(np.argmin(np.abs(arr - t)))
        return float(arr[j] - t) if abs(arr[j] - t) <= window_s else np.nan

    e_in = nearest(np.asarray(suna_entries, float), ctd_in)
    e_out = nearest(np.asarray(suna_exits, float), ctd_out)
    vals = [v for v in (e_in, e_out) if np.isfinite(v)]
    mean = float(np.mean(vals)) if vals else np.nan
    half_diff = float((e_in - e_out) / 2) if np.isfinite(e_in) and np.isfinite(e_out) else np.nan
    return dict(entry_offset_s=e_in, exit_offset_s=e_out, clock_estimate_s=mean,
                mount_timing_s=half_diff)


In [ ]:
# Known-answer validation: build casts whose lag is known, run every method, and check each one.
_rng = np.random.default_rng(3)
_dt, _w = 1.06, 0.6
_td = np.arange(0, 150 / _w, _dt)
_t = np.concatenate([_td, _td[-1] + 20 + _td])
_P = np.clip(np.concatenate([_w * _td, 150 - _w * _td]), 0, None)
_N = lambda p: 5 + 20 * (1 + np.tanh((p - 50) / 8)) / 2          # nitracline at 50 dbar
_TRUE = 10.0
_delay = _N(np.interp(_t - _TRUE, _t, _P)) + _rng.normal(0, 0.15, _t.size)
_x = _N(_P); _resp = np.empty_like(_x); _resp[0] = _x[0]
for _i in range(1, _x.size):
    _a = 1 - np.exp(-(_t[_i] - _t[_i - 1]) / _TRUE); _resp[_i] = _resp[_i - 1] + _a * (_x[_i] - _resp[_i - 1])
_resp += _rng.normal(0, 0.15, _t.size)

_rows = []
for _name, _n in (("pure 10 s delay (clock-like)", _delay), ("10 s first-order response", _resp)):
    _c = collapse_lag(_t, _n, _P, DT_S, MAX_LAG_S, BIN_DB)
    _xl, _xr = xcorr_lag(_t, _n, _P, DT_S, MAX_LAG_S)
    _rows.append({"synthetic case (truth = +10 s)": _name, "C collapse (s)": _c["lag"],
                  "hysteresis removed (%)": round(_c["pct_reduction"]), "B xcorr (s)": _xl,
                  "A peak-to-peak (s)": round(peak_lag(_t, _n, _P), 1)})
_val = pd.DataFrame(_rows); display(_val)
assert all(abs(_val["C collapse (s)"] - _TRUE) <= 1.5), "collapse method failed validation"

# Event check: CTD wet 100 to 900 s; SUNA clock 10 s late and mounted higher (3 s), with surface flicker.
_tt = np.arange(0, 1000, 0.25); _cond = np.where((_tt >= 100) & (_tt <= 900), 5.0, 0.02)
_ts = np.arange(0, 1000, 1.06); _wet = (_ts >= 113) & (_ts <= 907)
_fl = ((_ts > 113) & (_ts < 133)) | ((_ts > 887) & (_ts < 907)); _wet[_fl] = _rng.random(_fl.sum()) > 0.3
_ev = event_offsets(*ctd_water_events(_tt, _cond), *suna_water_transitions(_ts, ~_wet))
print(f"event check: clock {_ev['clock_estimate_s']:.1f} s (truth 10), mounting {_ev['mount_timing_s']:.1f} s (truth 3)")
assert abs(_ev["clock_estimate_s"] - 10) <= 1.5 and abs(_ev["mount_timing_s"] - 3) <= 1.5
print("validation passed: method C and the event check recover known lags; A and B do not, as expected.")

In [ ]:
def pick(df, *cands):
    "First column present, exact then case-insensitive (mirrors suncat.util.pick)."
    low = {c.lower(): c for c in df.columns}
    for c in cands:
        if c in df.columns: return c
        if c.lower() in low: return low[c.lower()]
    raise KeyError(f"none of {cands} in {list(df.columns)}")

if SOURCE == "colocated":
    raw = pd.read_csv(COLOCATED)
else:
    if not MERGED:
        raise FileNotFoundError("verify mode: no L2/SUNA/*_SUNA_1s.csv found; run step02_merge_ctd_suna first")
    raw = pd.concat([pd.read_csv(f, encoding="utf-8-sig") for f in MERGED], ignore_index=True)
df = pd.DataFrame({
    "time": pd.to_datetime(raw[pick(raw, "time_utc", "utc_time", "time")], errors="coerce"),
    "no3":  pd.to_numeric(raw[pick(raw, "nitrate_uM", "suna_nitrate", "no3_uM", "nitrate")], errors="coerce"),
    "pres": pd.to_numeric(raw[pick(raw, "pressure_db", "prdM", "prDM", "pressure_dbar", "pressure")], errors="coerce"),
    "cast": raw[pick(raw, "cast", "cast_id")].astype(str),
})
_flag = next((c for c in ("L1_clean", "suna_L1_clean") if c in raw.columns), None)
df["clean"] = raw[_flag].astype("boolean").fillna(False).astype(bool) if _flag else True
if "in_cast" in raw.columns:
    df = df[raw["in_cast"].astype(bool).to_numpy()]
df = df.dropna(subset=["time", "no3", "pres"]).sort_values(["cast", "time"]).reset_index(drop=True)
CASTS = sorted(df["cast"].unique())

def cast_arrays(cast):
    d = df[df["cast"] == cast]
    if USE_CLEAN and d["clean"].any():
        d = d[d["clean"]]
    t0 = d["time"].iloc[0]
    return (d["time"] - t0).dt.total_seconds().to_numpy(), d["no3"].to_numpy(), d["pres"].to_numpy(), t0, len(d)

print(f"{len(df)} frames, {len(CASTS)} casts: {', '.join(CASTS)}")

In [ ]:
def ascent_speed(t, p):
    o = np.argsort(t); t, p = t[o], p[o]; ib = int(np.argmax(p))
    w = -np.diff(p[ib:]) / np.diff(t[ib:]); w = w[np.isfinite(w) & (w > 0)]
    return float(np.median(w)) if w.size else np.nan

rows, CURVES = [], {}
for cast in CASTS:
    t, n, p, t0, nfr = cast_arrays(cast)
    if nfr < MIN_FRAMES:
        print(f"skip {cast}: only {nfr} usable frames"); continue
    lags, rms, _ = collapse_curve(t, n, p, DT_S, MAX_LAG_S, BIN_DB)
    c = collapse_lag(t, n, p, DT_S, MAX_LAG_S, BIN_DB)
    hat, lo, hi = collapse_lag_ci(t, n, p, DT_S, MAX_LAG_S, BIN_DB, BLOCK_BINS, N_BOOT)
    xl, xr = xcorr_lag(t, n, p, DT_S, MAX_LAG_S)
    wd, wu = descent_speed(t, p), ascent_speed(t, p)
    CURVES[cast] = (lags, rms, lo, hi)
    rows.append({
        "cast": cast, "start_utc": t0, "frames": nfr, "max_p_db": round(float(np.nanmax(p)), 1),
        "lag_s": hat, "ci95_lo_s": lo, "ci95_hi_s": hi,
        "rms_zero_uM": round(c["rms_zero"], 3), "rms_best_uM": round(c["rms_min"], 3),
        "hysteresis_removed_pct": round(c["pct_reduction"], 1),
        "w_down_dbar_s": round(wd, 3), "w_up_dbar_s": round(wu, 3),
        "leg_separation_db": round((wd + wu) * hat, 1) if np.isfinite(hat) else np.nan,
        "xcorr_lag_s": xl, "xcorr_r": round(xr, 3), "peak_lag_s": round(peak_lag(t, n, p), 1),
    })
STATS = pd.DataFrame(rows)
display(STATS)

In [ ]:
lag = STATS["lag_s"].to_numpy(dtype=float)
ok = np.isfinite(lag)
med = float(np.median(lag[ok])) if ok.any() else np.nan
mlo, mhi = bootstrap_median_ci(lag[ok])
q1, q3 = (np.percentile(lag[ok], [25, 75]) if ok.sum() >= 2 else (np.nan, np.nan))
excl0 = int(((STATS["ci95_lo_s"] > 0) | (STATS["ci95_hi_s"] < 0)).sum())

# Clock drift: least-squares slope of lag against cast time (hours since the first cast).
hrs = (pd.to_datetime(STATS["start_utc"]) - pd.to_datetime(STATS["start_utc"]).min()).dt.total_seconds() / 3600
drift = dict(slope_s_per_h=np.nan, intercept_s=np.nan, se=np.nan, p=np.nan)
if ok.sum() >= 3 and np.ptp(hrs[ok]) > 0:
    x, y = hrs[ok].to_numpy(), lag[ok]
    b, a = np.polyfit(x, y, 1)
    resid = y - (a + b * x); sxx = ((x - x.mean()) ** 2).sum()
    se = float(np.sqrt((resid ** 2).sum() / (len(x) - 2) / sxx))
    try:
        from scipy import stats as _st
        pval = float(2 * _st.t.sf(abs(b / se), len(x) - 2)) if se > 0 else np.nan
    except Exception:
        pval = np.nan
    drift = dict(slope_s_per_h=float(b), intercept_s=float(a), se=se, p=pval)

REC = -round(med * 2) / 2 if np.isfinite(med) else np.nan   # nearest 0.5 s, opposite sign (see convention)
CRUISE_STATS = pd.DataFrame([{
    "cruise": CRUISE_ID, "casts": int(ok.sum()), "median_lag_s": med, "median_ci95_lo_s": mlo,
    "median_ci95_hi_s": mhi, "iqr_lo_s": q1, "iqr_hi_s": q3,
    "min_lag_s": float(np.min(lag[ok])) if ok.any() else np.nan,
    "max_lag_s": float(np.max(lag[ok])) if ok.any() else np.nan,
    "casts_ci_excludes_zero": excl0, "median_hysteresis_removed_pct": float(STATS["hysteresis_removed_pct"].median()),
    "drift_s_per_h": drift["slope_s_per_h"], "drift_se": drift["se"], "drift_p": drift["p"],
    "recommended_SUNA_CLOCK_OFFSET_S": REC,
}])
display(CRUISE_STATS.T.rename(columns={0: "value"}))
print(f"\nCruise median lag {med:+.1f} s (95% CI {mlo:+.1f} to {mhi:+.1f} s, n = {ok.sum()} casts); "
      f"{excl0} of {ok.sum()} casts have a CI that excludes zero.")
if np.isfinite(drift["slope_s_per_h"]):
    print(f"Drift: {drift['slope_s_per_h']:+.3f} +/- {drift['se']:.3f} s per hour (p = {drift['p']:.3g}).")

In [ ]:
# Independent clock check from water entry and exit. Skips cleanly if the L1 inputs are not found.
def parse_cnv(path):
    "CTD .cnv -> DataFrame(time_utc, pressure_db, cond). Same parser as step02_suna_colocation."
    lines = Path(path).read_text(errors="replace").splitlines()
    names, start, iend = [], None, None
    for i, ln in enumerate(lines):
        if ln.startswith("# name"):
            names.append(ln.split("=", 1)[1].split(":")[0].strip())
        elif ln.startswith("# start_time"):
            s = ln.split("=", 1)[1].split("[")[0].strip()
            for fmt in ("%b %d %Y %H:%M:%S", "%b %d %Y  %H:%M:%S"):
                try: start = datetime.strptime(s, fmt); break
                except ValueError: pass
        elif ln.strip() == "*END*":
            iend = i + 1; break
    data = []
    for ln in lines[iend:]:
        q = ln.split()
        if len(q) >= len(names):
            try: data.append([float(x) for x in q[:len(names)]])
            except ValueError: pass
    arr = np.array(data, float)
    arr[(arr <= -8e-29) & (arr >= -1.2e-28)] = np.nan
    d = pd.DataFrame(arr, columns=names)
    def find(subs):
        for s in subs:
            for c in names:
                if c.lower().startswith(s) or s in c.lower(): return c
        return None
    tc, pc, cc = find(["times", "time, elapsed"]), find(["prdm", "pressure"]), find(["c0s/m", "c0ms/cm", "cond"])
    if start is None or tc is None or cc is None:
        raise ValueError(f"missing start_time/time/conductivity (start={start}, t={tc}, c={cc})")
    return pd.DataFrame({"time_utc": [start + timedelta(seconds=s) for s in d[tc].to_numpy()],
                         "cond": d[cc].to_numpy()})

def epoch(x):
    s = pd.to_datetime(pd.Series(x), errors="coerce")
    if getattr(s.dt, "tz", None) is not None:
        s = s.dt.tz_convert("UTC").dt.tz_localize(None)
    return (s - pd.Timestamp("1970-01-01")).dt.total_seconds().to_numpy()

EVENTS = pd.DataFrame()
cnvs = sorted(pp.product_dir(CRUISE_ID, "ctd_datcnv").glob("*.cnv"))
suna_tables = []
for root in (pp.product_dir(CRUISE_ID, "suna_l1"), CRUISE / "_audit" / "suna"):
    for f in sorted(root.rglob("*.csv")) if root.exists() else []:
        try:
            head = f.open(encoding="utf-8-sig").readline()
        except Exception:
            continue
        if "f_outofwater" in head and "time_utc" in head:
            suna_tables.append(f)

if SOURCE != "colocated":
    print("event check skipped in verify mode: it measures the raw clocks, which the merge correction does not change.")
elif not cnvs or not suna_tables:
    print(f"event check skipped: found {len(cnvs)} L1 CTD .cnv and {len(suna_tables)} SUNA tables with f_outofwater.")
else:
    sl = pd.concat([pd.read_csv(f, usecols=["time_utc", "f_outofwater"], encoding="utf-8-sig") for f in suna_tables])
    s_t = epoch(sl["time_utc"]); s_f = sl["f_outofwater"].astype(bool).to_numpy()
    entries, exits = suna_water_transitions(s_t, s_f)
    ev_rows = []
    for f in cnvs:
        try:
            c = parse_cnv(f)
        except Exception as e:
            print("  skip", f.name, "-", e); continue
        t_in, t_out = ctd_water_events(epoch(c["time_utc"]), c["cond"].to_numpy())
        ev = event_offsets(t_in, t_out, entries, exits, EVENT_WINDOW_S)
        ev_rows.append({"cast": f.stem, **ev})
    EVENTS = pd.DataFrame(ev_rows)
    display(EVENTS)
    if EVENTS["clock_estimate_s"].notna().any():
        print(f"event-based clock offset, cruise median: {EVENTS['clock_estimate_s'].median():+.1f} s "
              f"(compare with the collapse lag median {med:+.1f} s)")

In [ ]:
def ctd_predicted(cast):
    "Nitrate the SUNA should read with no lag: the collapsed (best-shift) profile evaluated at the CTD pressure."
    t, n, p, _, _ = cast_arrays(cast)
    best = STATS.set_index("cast").loc[cast, "lag_s"]
    c, dn, up = _profiles_after_shift(t, n, p, best, BIN_DB)
    ref = np.nanmean(np.vstack([dn, up]), axis=0)
    ok = np.isfinite(ref)
    return np.interp(p, c[ok], ref[ok], left=np.nan, right=np.nan)

def draw_cast(cast, offset=None):
    "Five panels: time series, nitracline zoom, legs at zero shift, legs after shift, cost curve."
    t, n, p, t0, _ = cast_arrays(cast)
    lags, rms, lo, hi = CURVES[cast]
    row = STATS.set_index("cast").loc[cast]
    L = row["lag_s"] if offset is None else offset
    fig = plt.figure(figsize=(16, 9.5)); gs = fig.add_gridspec(2, 3)
    a1 = fig.add_subplot(gs[0, 0:2]); a5 = fig.add_subplot(gs[0, 2])
    a2, a3, a4 = (fig.add_subplot(gs[1, k]) for k in range(3))
    tt = t0 + pd.to_timedelta(t, unit="s")
    a1.plot(tt, n, color="tab:blue", alpha=0.3, lw=1, label="SUNA nitrate, as logged")
    a1.plot(tt - pd.to_timedelta(L, unit="s"), n, color="tab:blue", lw=1.3, label=f"SUNA nitrate shifted {-L:+.1f} s")
    a1.set_ylabel("Raw nitrate (umol L-1)", color="tab:blue"); a1.legend(loc="center left", fontsize=8)
    b1 = a1.twinx(); b1.plot(tt, p, color="tab:orange", lw=1.2); b1.invert_yaxis()
    b1.set_ylabel("Pressure (dbar)", color="tab:orange")
    a1.set_title(f"{cast}: nitrate and depth vs time"); a1.tick_params(axis="x", rotation=30)

    # Zoom on the downcast nitracline, where a lag is easiest to see.
    pred = ctd_predicted(cast)
    o = np.argsort(t); ib = int(np.argmax(p[o])); down = o[: ib + 1]
    g = np.gradient(np.nan_to_num(pred[down], nan=np.nanmedian(pred)), t[down])
    ts_ = t[down][int(np.nanargmax(g))]; w = (t >= ts_ - 75) & (t <= ts_ + 75)
    a5.plot(t[w] - ts_, pred[w], "k--", lw=1.4, label="CTD-predicted (no lag)")
    a5.plot(t[w] - ts_, n[w], color="tab:blue", alpha=0.35, lw=1.2, label="SUNA as logged")
    a5.plot(t[w] - L - ts_, n[w], color="tab:blue", lw=1.5, label=f"SUNA shifted {-L:+.1f} s")
    a5.set_xlim(-75, 75); a5.set_xlabel("Seconds from steepest downcast nitrate rise")
    a5.set_ylabel("Raw nitrate (umol L-1)"); a5.legend(fontsize=8, loc="upper left")
    a5.set_title(f"zoom: SUNA as logged arrives {row['lag_s']:+.1f} s late")
    for ax, lag_used, ttl in ((a2, 0.0, "legs as logged (zero shift)"), (a3, L, f"legs after shift of {-L:+.1f} s")):
        c, dn, up = _profiles_after_shift(t, n, p, lag_used, BIN_DB)
        if c is not None:
            ax.plot(dn, c, color="tab:blue", lw=1.4, label="down")
            ax.plot(up, c, color="tab:orange", lw=1.4, label="up")
        ax.invert_yaxis(); ax.set_xlabel("Raw nitrate (umol L-1)"); ax.set_ylabel("Pressure (dbar)")
        ax.set_title(ttl); ax.legend(fontsize=8)
    a4.plot(lags, rms, color="k", lw=1.3)
    if np.isfinite(lo) and np.isfinite(hi):
        a4.axvspan(lo, hi, color="tab:green", alpha=0.2, label=f"95% CI [{lo:+.1f}, {hi:+.1f}] s")
    a4.axvline(row["lag_s"], color="tab:green", lw=1.5, label=f"best lag {row['lag_s']:+.1f} s")
    a4.axvline(0, color="grey", ls="--", lw=1)
    if offset is not None:
        a4.axvline(offset, color="tab:red", lw=1.5, label=f"slider {offset:+.1f} s")
    a4.set_xlabel("Lag (s, positive = SUNA late)"); a4.set_ylabel("Down/up RMS difference (umol L-1)")
    a4.set_title(f"cost curve: {row['hysteresis_removed_pct']:.0f}% of hysteresis removed at best lag")
    a4.legend(fontsize=8)
    fig.tight_layout()
    return fig

for cast in STATS["cast"]:
    draw_cast(cast).savefig(OUT / f"{cast}__time_lag{TAG}.png", dpi=150, bbox_inches="tight"); plt.close("all")
draw_cast(STATS["cast"].iloc[0]); plt.show()

In [ ]:
fig, (ax, bx) = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [2, 1]})
yerr = np.vstack([STATS["lag_s"] - STATS["ci95_lo_s"], STATS["ci95_hi_s"] - STATS["lag_s"]])
ax.errorbar(hrs, STATS["lag_s"], yerr=yerr, fmt="o", color="tab:blue", capsize=3, label="collapse lag, 95% CI")
if np.isfinite(med):
    ax.axhline(med, color="tab:blue", lw=1); ax.axhspan(mlo, mhi, color="tab:blue", alpha=0.12, label=f"cruise median {med:+.1f} s")
if np.isfinite(drift["slope_s_per_h"]):
    xx = np.linspace(hrs.min(), hrs.max(), 50)
    ax.plot(xx, drift["intercept_s"] + drift["slope_s_per_h"] * xx, "k--", lw=1,
            label=f"drift {drift['slope_s_per_h']:+.2f} s/h (p={drift['p']:.2g})")
if not EVENTS.empty and EVENTS["clock_estimate_s"].notna().any():
    e = STATS[["cast"]].merge(EVENTS, on="cast", how="left")
    ax.plot(hrs, e["clock_estimate_s"], "s", mfc="none", color="tab:red", label="event-based clock")
ax.axhline(0, color="grey", ls=":", lw=1)
for x, y, c in zip(hrs, STATS["lag_s"], STATS["cast"]):
    ax.annotate(c.split("_")[-1], (x, y), textcoords="offset points", xytext=(4, 4), fontsize=7)
ax.set_xlabel("Hours since first cast"); ax.set_ylabel("SUNA lag (s, positive = SUNA late)")
ax.set_title(f"{CRUISE_ID}: SUNA to CTD lag by cast"); ax.legend(fontsize=8)
bx.bar(range(len(STATS)), STATS["hysteresis_removed_pct"], color="tab:green")
bx.set_xticks(range(len(STATS))); bx.set_xticklabels([c.split("_")[-1] for c in STATS["cast"]], fontsize=8)
bx.set_ylabel("% of down/up mismatch removed"); bx.set_ylim(0, 100); bx.set_title("how much a time shift explains")
plt.tight_layout(); plt.savefig(OUT / f"{CRUISE_ID}__time_lag_summary{TAG}.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Interactive window: pick a cast and drag the slider to shift the SUNA time. At the best lag the two
# legs collapse, the shifted trace sits on the CTD-predicted curve, and the red line reaches the bottom
# of the cost curve.
def draw_interactive(cast, lag_s):
    plt.close("all"); draw_cast(cast, offset=lag_s)
    lags, rms, _, _ = CURVES[cast]
    r, r0 = rms[np.argmin(np.abs(lags - lag_s))], rms[np.argmin(np.abs(lags))]
    pct = 100 * (1 - r / r0)
    rel = f"{pct:.0f}% below zero shift" if pct >= 0 else f"{-pct:.0f}% worse than zero shift"
    print(f"{cast}: at lag {lag_s:+.1f} s the down/up RMS is {r:.2f} umol L-1 ({rel}). "
          f"step02 setting for this lag: {(-lag_s or 0.0):+.1f} s")
    plt.show()

try:
    import ipywidgets as W
    dd = W.Dropdown(options=list(STATS["cast"]), description="cast")
    sl = W.FloatSlider(value=float(STATS["lag_s"].iloc[0]), min=-MAX_LAG_S, max=MAX_LAG_S, step=DT_S,
                       description="lag (s)", continuous_update=False, readout_format="+.1f")
    snap = W.Button(description="snap to best lag")
    def _cast_changed(ch):
        v = STATS.set_index("cast").loc[ch["new"], "lag_s"]; sl.value = float(v) if np.isfinite(v) else 0.0
    dd.observe(_cast_changed, names="value")
    snap.on_click(lambda _: setattr(sl, "value", float(STATS.set_index("cast").loc[dd.value, "lag_s"])))
    out = W.interactive_output(draw_interactive, {"cast": dd, "lag_s": sl})
    display(W.VBox([W.HBox([dd, sl, snap]), out]))
except Exception as e:
    print("ipywidgets not available; the static figures are saved in", OUT, "-", e)

In [ ]:
STATS.to_csv(OUT / f"{CRUISE_ID}__suna_time_lag_per_cast{TAG}.csv", index=False)
CRUISE_STATS.to_csv(OUT / f"{CRUISE_ID}__suna_time_lag_cruise{TAG}.csv", index=False)
if not EVENTS.empty:
    EVENTS.to_csv(OUT / f"{CRUISE_ID}__suna_clock_events.csv", index=False)
print("saved to", OUT, "\n")

if SOURCE == "colocated":
    # Proposed per-cast offsets for step02. With no significant drift, every cast gets the cruise median
    # (less noisy than single-cast values); with significant drift (p < 0.05, at least 4 casts), each cast
    # gets the drift-fit value at its time. offset = -lag, rounded to 0.5 s.
    use_drift = np.isfinite(drift["p"]) and drift["p"] < 0.05 and ok.sum() >= 4
    if use_drift:
        lag_used = drift["intercept_s"] + drift["slope_s_per_h"] * hrs.to_numpy()
        basis = f"drift fit ({drift['slope_s_per_h']:+.2f} s/h, p={drift['p']:.2g})"
    else:
        lag_used = np.full(len(STATS), med)
        basis = "cruise median (no significant drift)"
    PROPOSED = pd.DataFrame({
        "cast_id": STATS["cast"],
        "offset_s": [(-round(v * 2) / 2) or 0.0 for v in lag_used],
        "measured_lag_s": STATS["lag_s"], "ci95_lo_s": STATS["ci95_lo_s"], "ci95_hi_s": STATS["ci95_hi_s"],
        "basis": basis,
    })
    prop_path = OUT / f"{CRUISE_ID}__suna_time_offsets_PROPOSED.csv"
    PROPOSED.to_csv(prop_path, index=False)
    display(PROPOSED)
    print("PROPOSED OFFSETS FOR step02 (not applied until you confirm them)")
    print(f"  measured: SUNA lags the CTD by {med:+.1f} s (cruise median, 95% CI {mlo:+.1f} to {mhi:+.1f} s); basis: {basis}")
    print(f"  written : {prop_path}")
    print(f"  confirm : review it, then save as {CRUISE / 'metadata' / 'suna_time_offsets.csv'}")
    print('  apply   : in step02_merge_ctd_suna set TIME_OFFSET_MODE = "table", re-run step02 and step03')
    print('  verify  : re-run this notebook with SOURCE = "merged"; the residual lag should be about zero')
    print("  keep step02_suna_colocation at zero offset, so this notebook keeps measuring the raw lag")
else:
    verified = np.isfinite(med) and abs(med) <= 1.5 and mlo <= 0 <= mhi
    print("VERIFICATION OF THE step02 CORRECTION")
    print(f"  residual lag: {med:+.1f} s (cruise median, 95% CI {mlo:+.1f} to {mhi:+.1f} s, n = {ok.sum()} casts)")
    print("  verdict     : " + ("correction verified, residual within about +/-1.5 s and its CI includes zero"
                                if verified else
                                "residual remains; check the offsets table, its sign, and TIME_OFFSET_MODE in step02"))

## How to read the results

- **Best lag clearly non-zero, CIs exclude zero, little spread across casts, no significant drift:** one
  constant time offset explains the mismatch. Apply the recommended setting in step02.
- **Event-based clock agrees with the collapse lag:** the delay is a clock offset. **Event clock near zero but
  collapse lag clearly positive:** the delay is in the sensor or its plumbing (response time), not the clock.
  A time shift is still the right first correction, but a response-time correction may also be worth it.
- **Significant drift:** the clocks diverge through the cruise; apply a time-varying offset, not a constant.
- **Share of hysteresis removed:** close to 100% means a pure shift explains the mismatch. A clearly lower share
  means smoothing (response time) and/or real differences between the legs (drift, internal waves).
- **Mounting:** a vertical offset between the SUNA and the CTD pressure sensor gives a constant depth offset that
  none of this detects. It needs the rosette geometry, which is why the photo matters.

**Resolution:** the SUNA records about one frame per second, so lags are resolved to about 1 s; the bootstrap
CIs carry the profile-noise uncertainty on top of that. The event check has two further limits worth stating:
the SBE `.cnv` header records `start_time` to the whole second (up to 1 s), and an event is detected at the
first SUNA frame on or after it (up to one frame, about 1 s, late). Together the event-based clock is good to
about +/-1 to 2 s, enough to tell a clock offset of ~10 s from zero, not to split hairs below that.

**Effect downstream:** step03 matches bottles to SUNA frames by depth on both legs. An uncorrected lag puts the
downcast frames too deep and the upcast frames too shallow, which spreads the bottle comparison across the
nitracline. Correcting the lag before the bottle calibration should tighten that fit.

## References

- Bittig, H. C., Fiedler, B., Scholz, R., Krahmann, G., & Körtzinger, A. (2014). Time response of oxygen optodes on
  profiling platforms and its dependence on flow speed and temperature. *Limnology and Oceanography: Methods*, 12,
  617-636. https://doi.org/10.4319/lom.2014.12.617
- Bittig, H. C., & Körtzinger, A. (2017). Technical note: Update on response times, in-air measurements, and in situ
  drift for oxygen optodes on profiling platforms. *Ocean Science*, 13, 1-11. https://doi.org/10.5194/os-13-1-2017
- Bittig, H. C., et al. (2018). Oxygen optode sensors: principle, characterization, calibration, and application in
  the ocean. *Frontiers in Marine Science*, 4, 429. https://doi.org/10.3389/fmars.2017.00429
- Efron, B., & Tibshirani, R. J. (1993). *An Introduction to the Bootstrap*. Chapman & Hall.
- Garau, B., Ruiz, S., Zhang, W. G., Pascual, A., Heslop, E., Kerfoot, J., & Tintoré, J. (2011). Thermal lag
  correction on Slocum CTD glider data. *Journal of Atmospheric and Oceanic Technology*, 28, 1065-1071.
  https://doi.org/10.1175/JTECH-D-10-05030.1
- Johnson, K. S., & Coletti, L. J. (2002). In situ ultraviolet spectrophotometry for high resolution and long-term
  monitoring of nitrate, bromide and bisulfide in the ocean. *Deep-Sea Research I*, 49, 1291-1305.
  https://doi.org/10.1016/S0967-0637(02)00020-1
- Künsch, H. R. (1989). The jackknife and the bootstrap for general stationary observations. *Annals of
  Statistics*, 17, 1217-1241.
- Sakamoto, C. M., Johnson, K. S., & Coletti, L. J. (2009). Improved algorithm for the computation of nitrate
  concentrations in seawater using an in situ ultraviolet spectrophotometer. *Limnology and Oceanography:
  Methods*, 7, 132-143.
- Thomson, R. E., & Emery, W. J. (2014). *Data Analysis Methods in Physical Oceanography* (3rd ed.). Elsevier.
- Sea-Bird Scientific. SBE Data Processing manual (Align CTD module: advancing sensors in time relative to
  pressure), and SUNA V2 user manual.